In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Paths
DATA_ROOT = "/content/drive/MyDrive/400x400_paired_data"
SAVE_DIR = "/content/drive/MyDrive/vae_finetuning/models"
!mkdir -p "$SAVE_DIR"

# Dependencies
!pip install -q diffusers==0.30.0 transformers accelerate torch torchvision tqdm pillow termcolor

# Imports
import os
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from diffusers import AutoencoderKL
from termcolor import colored

# config
EPOCHS = 10
BATCH_SIZE = 2              # lowered for diagnostics (paper uses 16)
LR = 1e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GRAD_CLIP = 1.0


# Dataset

class WatermarkDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.tf = transform
        all_folders = sorted([d for d in os.listdir(root) if not d.startswith(".")])
        self.pairs = []
        for folder in all_folders:
            fpath = os.path.join(root, folder)
            if not os.path.isdir(fpath):
                continue
            xw = os.path.join(fpath, f"{folder}_hidden.png")
            xi = os.path.join(fpath, f"{folder}_hidden_inv.png")
            if os.path.exists(xw) and os.path.exists(xi):
                self.pairs.append((xw, xi))
        print(f"Loaded {len(self.pairs)} paired samples from {len(all_folders)} folders.")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        xw_path, xi_path = self.pairs[idx]
        xw = Image.open(xw_path).convert("RGB")
        xi = Image.open(xi_path).convert("RGB")
        if self.tf:
            xw = self.tf(xw)
            xi = self.tf(xi)
        return xw, xi


# --- Transform ---
transform = transforms.Compose([
    transforms.Resize((400, 400)),
    transforms.ToTensor(),   # range [0, 1]
])

dataset = WatermarkDataset(DATA_ROOT, transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

# Model
vae = AutoencoderKL.from_pretrained("stabilityai/sdxl-vae").to(DEVICE)
vae.train()
optimizer = torch.optim.Adam(vae.parameters(), lr=LR)
criterion = nn.MSELoss()

# Diagnostics Helper

def tensor_stats(name, t):
    if t is None:
        print(f"{name:12s} | None")
        return
    with torch.no_grad():
        nan = torch.isnan(t).any().item()
        inf = torch.isinf(t).any().item()
        msg = (
            f"{name:12s} | shape={tuple(t.shape)} | dtype={t.dtype} "
            f"| min={t.min().item():.4f}, max={t.max().item():.4f}, "
            f"mean={t.mean().item():.4f}, std={t.std().item():.4f}, nan={nan}, inf={inf}"
        )
        print(colored(msg, "red") if (nan or inf) else msg)


# Finetuning Loop (Algorithm 1 + diagnostics)

print(colored("Starting adaptive VAE fine-tuning (diagnostic mode)", "cyan"))

for epoch in range(EPOCHS):
    vae.train()
    total_loss = 0.0

    for step, (xw, xi) in enumerate(tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}"), start=1):
        xw, xi = xw.to(DEVICE, non_blocking=True), xi.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        # --- Input stats ---
        if step == 1:
            print("\n=== INPUT CHECK ===")
            tensor_stats("xw", xw)
            tensor_stats("xi", xi)

        # --- Encode + Decode ---
        try:
            posterior = vae.encode(xw)
            z = posterior.latent_dist.mean
            tensor_stats("z_mean", z)

            # Decode (paper step 6)
            x_inv_hat = vae.decode(z / vae.config.scaling_factor).sample
            tensor_stats("x_inv_hat", x_inv_hat)
        except Exception as e:
            print(colored(f"Forward crash: {e}", "red"))
            break

        # --- Compute loss (paper Eq. 1) ---
        try:
            loss = criterion(x_inv_hat, xi)
        except Exception as e:
            print(colored(f"Loss computation crash: {e}", "red"))
            break

        # --- Backpropagation (Algorithm 1, step 9) ---
        if torch.isnan(loss):
            print(colored("NaN loss detected – skipping batch", "red"))
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(vae.parameters(), GRAD_CLIP)
        optimizer.step()

        total_loss += loss.item() * xw.size(0)

        if step % 10 == 0:
            print(f"[Epoch {epoch+1} | Step {step}] Loss = {loss.item():.6f}")

        if torch.isnan(z).any() or torch.isnan(x_inv_hat).any():
            print(colored(f"NaN detected in latent or output at step {step}", "red"))
            raise SystemExit

    avg_loss = total_loss / len(dataset)
    print(colored(f"Epoch [{epoch+1}/{EPOCHS}] - Avg MSE Loss: {avg_loss:.6f}", "cyan"))

    # --- Save sample reconstructions ---
    with torch.no_grad():
        xw_vis, xi_vis = dataset[0]
        xw_vis = xw_vis.unsqueeze(0).to(DEVICE)
        posterior_vis = vae.encode(xw_vis)
        z_vis = posterior_vis.latent_dist.mean
        x_hat_vis = vae.decode(z_vis / vae.config.scaling_factor).sample

        def denorm(x): return x.clamp(0, 1)
        save_image(torch.cat([
          denorm(xw_vis),
          denorm(x_hat_vis),
          denorm(xi_vis.unsqueeze(0).to(DEVICE))
        ], dim=0),
        os.path.join(SAVE_DIR, f"epoch{epoch+1}_recon.png"), nrow=3)
        print(f"Saved reconstruction preview → epoch{epoch+1}_recon.png")


# Save final model

vae.save_pretrained(SAVE_DIR)
print(colored(f"Finetuning complete! Model saved to {SAVE_DIR}", "green"))


In [ ]:
import tensorflow as tf
tf.compat.v1.disable_eager_execution()

def list_signatures_fixed(model_dir):
    g = tf.Graph()
    with tf.compat.v1.Session(graph=g) as s:
        meta = tf.compat.v1.saved_model.loader.load(s, [tf.saved_model.SERVING], model_dir)
        print("Available signature keys:", list(meta.signature_def.keys()))
        for k, sig in meta.signature_def.items():
            print(f"\n== Signature: {k} ==")
            for name, tensor_info in sig.inputs.items():
                print(f"  Input:  {name:<15} | Tensor name: {tensor_info.name:<30} | dtype: {tensor_info.dtype:<10}")
            for name, tensor_info in sig.outputs.items():
                print(f"  Output: {name:<15} | Tensor name: {tensor_info.name:<30} | dtype: {tensor_info.dtype:<10}")

MODEL_DIR = "/content/drive/MyDrive/stegastamp_pretrained"
list_signatures_fixed(MODEL_DIR)


Available signature keys: ['serving_default']

== Signature: serving_default ==
  Input:  image           | Tensor name: input_hide:0                   | dtype: 1         
  Input:  secret          | Tensor name: input_prep:0                   | dtype: 1         
  Output: stegastamp      | Tensor name: clip_by_value:0                | dtype: 1         
  Output: decoded         | Tensor name: Round:0                        | dtype: 1         
  Output: residual        | Tensor name: gen_encoder/enc_conv_15/BiasAdd:0 | dtype: 1         


In [ ]:
import os, numpy as np, tensorflow as tf
tf.compat.v1.disable_eager_execution()

MODEL_DIR = "/content/drive/MyDrive/stegastamp_pretrained"

def load_decoder_handles(model_dir):
    g = tf.Graph()
    sess = tf.compat.v1.Session(graph=g)
    meta = tf.compat.v1.saved_model.loader.load(sess, [tf.saved_model.SERVING], model_dir)
    sig = meta.signature_def["serving_default"]

    inp_image_name  = sig.inputs["image"].name
    inp_secret_name = sig.inputs["secret"].name
    out_bits_name   = sig.outputs["decoded"].name

    inp_image  = g.get_tensor_by_name(inp_image_name)
    inp_secret = g.get_tensor_by_name(inp_secret_name)
    out_bits   = g.get_tensor_by_name(out_bits_name)

    return sess, {"image": inp_image, "secret": inp_secret}, out_bits

sess, inputs_dict, out_bits = load_decoder_handles(MODEL_DIR)
print("Decoder tensors ready:",
      inputs_dict["image"].name, inputs_dict["secret"].name, "->", out_bits.name)


✓ Decoder tensors ready: input_hide:0 input_prep:0 -> Round:0


In [ ]:
from PIL import Image, ImageOps
import glob

def preprocess_img(path):
    im = Image.open(path).convert("RGB")
    im = ImageOps.fit(im, (400, 400))
    arr = (np.asarray(im).astype(np.float32) / 255.0)
    return arr

def read_secret_bits(txt_path):
    with open(txt_path, "r") as f:
        s = f.read().strip()
    # supports "0101..." or "0 1 0 1 ..." or comma/space separated
    s = "".join(ch for ch in s if ch in "01")
    bits = np.fromiter((1 if c=='1' else 0 for c in s), dtype=np.int64)
    assert bits.size == 100, f"Secret length is {bits.size}, expected 100: {txt_path}"
    return bits

def decode_bits(sess, inputs_dict, out_tensor, img_path):
    arr = preprocess_img(img_path)
    dummy_secret = np.zeros((1,100), dtype=np.float32)  # required by signature, ignored by decoder path
    feed = {inputs_dict["image"]: arr[np.newaxis, ...], inputs_dict["secret"]: dummy_secret}
    pred = sess.run(out_tensor, feed_dict=feed)         # shape (1,100) or (100,)
    pred = np.squeeze(pred)
    # 'decoded' is already rounded (binary). If it weren’t, we’d do np.rint here.
    return pred.astype(np.int64)

def hamming(a, b):
    L = min(len(a), len(b))
    return int(np.sum(a[:L] != b[:L])), L


In [ ]:
DATA_ROOT   = "/content/drive/MyDrive/400x400_paired_data"
VAE_OUT_DIR = "/content/drive/MyDrive/vae_finetuning/outputs"   # where you saved *_vae.png

# Build a per-id dict of paths we’ll use:
items = []
for d in sorted(os.listdir(DATA_ROOT)):
    dpath = os.path.join(DATA_ROOT, d)
    if not os.path.isdir(dpath) or d.startswith("."):
        continue
    hid = os.path.join(dpath, f"{d}_hidden.png")
    sec = os.path.join(dpath, f"{d}_secret.txt")
    # Skip if missing
    if not (os.path.isfile(hid) and os.path.isfile(sec)):
        continue
    vae_png = os.path.join(VAE_OUT_DIR, f"{d}_vae.png")  # produced by your VAE eval cell
    items.append({
        "id": d,
        "hidden": hid,
        "secret": sec,
        "vae": vae_png if os.path.isfile(vae_png) else None
    })

print(f"Found {len(items)} items with hidden+secret; VAE outputs present for",
      sum(1 for it in items if it['vae'] is not None))


Found 1000 items with hidden+secret; VAE outputs present for 1000


In [ ]:
probe = next(it for it in items if it["vae"] is not None)  # pick one that has a VAE output
bits_hidden = decode_bits(sess, inputs_dict, out_bits, probe["hidden"])
bits_vae    = decode_bits(sess, inputs_dict, out_bits, probe["vae"])
gt          = read_secret_bits(probe["secret"])

ham_h, Lh = hamming(bits_hidden, gt)
ham_v, Lv = hamming(bits_vae,    gt)
print("ID:", probe["id"])
print("  hidden   Hamming:", ham_h, "/", Lh, "  (acc =", 1 - ham_h/Lh, ")")
print("  vae      Hamming:", ham_v, "/", Lv, "  (acc =", 1 - ham_v/Lv, ")")


ID: 000000
  hidden   Hamming: 0 / 100   (acc = 1.0 )
  vae      Hamming: 41 / 100   (acc = 0.5900000000000001 )


In [ ]:
import pandas as pd
from tqdm import tqdm

records = []
for it in tqdm(items):
    gt = read_secret_bits(it["secret"])

    bits_hidden = decode_bits(sess, inputs_dict, out_bits, it["hidden"])
    ham_h, Lh = hamming(bits_hidden, gt)

    ham_v, Lv, acc_v = None, None, None
    if it["vae"] is not None:
        bits_vae = decode_bits(sess, inputs_dict, out_bits, it["vae"])
        ham_v, Lv = hamming(bits_vae, gt)
        acc_v = 1 - ham_v / Lv

    records.append({
        "id": it["id"],
        "ham_hidden": ham_h,
        "acc_hidden": 1 - ham_h / Lh,
        "ham_vae": ham_v,
        "acc_vae": acc_v
    })

df = pd.DataFrame(records).sort_values("id")
out_csv = "/content/drive/MyDrive/vae_finetuning/eval_results.csv"
df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

print("\nSummary:")
print("Hidden acc (mean):", df["acc_hidden"].mean())
print("VAE acc (mean)   :", df["acc_vae"].dropna().mean())
df.head()


100%|██████████| 1000/1000 [06:47<00:00,  2.46it/s]

✅ Saved: /content/drive/MyDrive/vae_finetuning/eval_results.csv

Summary:
Hidden acc (mean): 0.99922
VAE acc (mean)   : 0.5453300000000001


,id,ham_hidden,acc_hidden,ham_vae,acc_vae
0,000000,0,1.00,41,0.59
1,000001,0,1.00,41,0.59
2,000002,1,0.99,59,0.41
3,000003,0,1.00,37,0.63
4,000004,0,1.00,33,0.67
